<a href="https://colab.research.google.com/github/danizaguirre/numpy/blob/main/s15_prueba_A_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import scipy.stats as stats
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta


In [6]:
# Objetivos del estudio
print("=== OBJETIVOS DEL ESTUDIO ===")
print("Objetivo principal: Evaluar el impacto de un sistema de recomendaciones mejorado")
print("Métricas clave:")
print("   • product_page: vistas de página de producto")
print("   • product_card: agregar al carrito")
print("   • purchase: compras")
print("Meta: ≥10% de mejora en cada etapa del embudo")
print("Audiencia: 15% nuevos usuarios UE (6,000 esperados)")
print("Período: 2020-12-07 a 2021-01-01")


=== OBJETIVOS DEL ESTUDIO ===
Objetivo principal: Evaluar el impacto de un sistema de recomendaciones mejorado
Métricas clave:
   • product_page: vistas de página de producto
   • product_card: agregar al carrito
   • purchase: compras
Meta: ≥10% de mejora en cada etapa del embudo
Audiencia: 15% nuevos usuarios UE (6,000 esperados)
Período: 2020-12-07 a 2021-01-01


In [8]:
# Cargar datasets
marketing_events = pd.read_csv('/ab_project_marketing_events_us.csv')
new_users = pd.read_csv('/final_ab_new_users_upd_us.csv')
events = pd.read_csv('/final_ab_events_upd_us.csv')
participants = pd.read_csv('/final_ab_participants_upd_us.csv')


In [11]:

# Exploración inicial
print("=== EXPLORACIÓN INICIAL ===")
print(f"📊 Marketing events: {marketing_events.shape}")
print(f"👥 New users: {new_users.shape}")
print(f"🎯 Events: {events.shape}")
print(f"🧪 Participants: {participants.shape}")
print()
print(f"📊 Marketing events: {marketing_events.info}")
print(f"👥 New users: {new_users.info}")
print(f"🎯 Events: {events.info}")
print(f"🧪 Participants: {participants.info}")


=== EXPLORACIÓN INICIAL ===
📊 Marketing events: (14, 4)
👥 New users: (58703, 4)
🎯 Events: (423761, 4)
🧪 Participants: (14525, 3)

📊 Marketing events: <bound method DataFrame.info of                                 name                   regions    start_dt  \
0           Christmas&New Year Promo             EU, N.America  2020-12-25   
1       St. Valentine's Day Giveaway  EU, CIS, APAC, N.America  2020-02-14   
2             St. Patric's Day Promo             EU, N.America  2020-03-17   
3                       Easter Promo  EU, CIS, APAC, N.America  2020-04-12   
4                  4th of July Promo                 N.America  2020-07-04   
5          Black Friday Ads Campaign  EU, CIS, APAC, N.America  2020-11-26   
6             Chinese New Year Promo                      APAC  2020-01-25   
7   Labor day (May 1st) Ads Campaign             EU, CIS, APAC  2020-05-01   
8    International Women's Day Promo             EU, CIS, APAC  2020-03-08   
9    Victory Day CIS (May 9th) Event  

In [12]:
# Veamos los formatos de fecha
print("=== FORMATOS DE FECHA ===")
print("Marketing events:")
print(marketing_events[['start_dt', 'finish_dt']].head(2))
print("\nNew users:")
print(new_users['first_date'].head(2))
print("\nEvents:")
print(events['event_dt'].head(2))

=== FORMATOS DE FECHA ===
Marketing events:
     start_dt   finish_dt
0  2020-12-25  2021-01-03
1  2020-02-14  2020-02-16

New users:
0    2020-12-07
1    2020-12-07
Name: first_date, dtype: object

Events:
0    2020-12-07 20:22:03
1    2020-12-07 09:22:53
Name: event_dt, dtype: object


In [13]:
# Primero, verifiquemos los tipos actuales
print("=== TIPOS DE DATOS ACTUALES ===")
for df_name, df in [('marketing_events', marketing_events),
                    ('new_users', new_users),
                    ('events', events),
                    ('participants', participants)]:
    print(f"\n{df_name}:")
    print(df.dtypes)

=== TIPOS DE DATOS ACTUALES ===

marketing_events:
name         object
regions      object
start_dt     object
finish_dt    object
dtype: object

new_users:
user_id       object
first_date    object
region        object
device        object
dtype: object

events:
user_id        object
event_dt       object
event_name     object
details       float64
dtype: object

participants:
user_id    object
group      object
ab_test    object
dtype: object


Conversión de tipos de datos

In [14]:
# 1. CONVERSIÓN DE FECHAS A DATETIME
print("=== CONVIRTIENDO FECHAS ===")

# Marketing events
marketing_events['start_dt'] = pd.to_datetime(marketing_events['start_dt'])
marketing_events['finish_dt'] = pd.to_datetime(marketing_events['finish_dt'])

# New users
new_users['first_date'] = pd.to_datetime(new_users['first_date'])

# Events (ya tiene timestamp, pero asegurémonos)
events['event_dt'] = pd.to_datetime(events['event_dt'])

print("✅ Fechas convertidas exitosamente")


=== CONVIRTIENDO FECHAS ===
✅ Fechas convertidas exitosamente


In [18]:
# 1. REVISAR VALORES AUSENTES EN CADA TABLA
print("=== ANÁLISIS DE VALORES AUSENTES ===")

# Marketing events
print("MARKETING EVENTS:")
print(marketing_events.info())
print("Valores ausentes por columna:")
print(marketing_events.isna().sum())
print()

# New users
print("NEW USERS:")
print(new_users.info())
print("Valores ausentes por columna:")
print(new_users.isna().sum())
print()

# Events
print("EVENTS:")
print(events.info())
print("Valores ausentes por columna:")
print(events.isna().sum())
print()

# Participants
print("PARTICIPANTS:")
print(participants.info())
print("Valores ausentes por columna:")
print(participants.isna().sum())

=== ANÁLISIS DE VALORES AUSENTES ===
MARKETING EVENTS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   name       14 non-null     object        
 1   regions    14 non-null     object        
 2   start_dt   14 non-null     datetime64[ns]
 3   finish_dt  14 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 580.0+ bytes
None
Valores ausentes por columna:
name         0
regions      0
start_dt     0
finish_dt    0
dtype: int64

NEW USERS:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     58703 non-null  object        
 1   first_date  58703 non-null  datetime64[ns]
 2   region      58703 non-null  object        
 3   device      58703 non-nu

In [19]:
# ANÁLISIS DE DUPLICADOS EN EVENTS
print("=== ANÁLISIS CORREGIDO DE DUPLICADOS ===")

# Usar la columna correcta (probablemente 'user_id')
print("Usuarios duplicados en events:")
print(f"Total eventos: {len(events)}")
print(f"Usuarios únicos: {events['user_id'].nunique()}")
print(f"¿Hay usuarios con múltiples eventos? {len(events) > events['user_id'].nunique()}")

# Eventos por usuario (muy importante para análisis de embudo)
events_per_user = events['user_id'].value_counts()
print(f"\nPromedio de eventos por usuario: {events_per_user.mean():.2f}")
print(f"Máximo eventos por usuario: {events_per_user.max()}")
print(f"Mínimo eventos por usuario: {events_per_user.min()}")

=== ANÁLISIS CORREGIDO DE DUPLICADOS ===
Usuarios duplicados en events:
Total eventos: 423761
Usuarios únicos: 58703
¿Hay usuarios con múltiples eventos? True

Promedio de eventos por usuario: 7.22
Máximo eventos por usuario: 36
Mínimo eventos por usuario: 1


CARACTERIZACIÓN DE LOS DATOS:
- No hay filas completamente duplicadas en ninguna tabla
- Los datos están bien estructurados

In [20]:
# CARACTERIZACIÓN DETALLADA DEL COMPORTAMIENTO
print("=== CARACTERIZACIÓN DEL COMPORTAMIENTO DE USUARIOS ===")

events_per_user = events['user_id'].value_counts()

# Distribución de eventos por usuario
print("Distribución de eventos por usuario:")
print(f"- Usuarios con 1 evento: {(events_per_user == 1).sum()}")
print(f"- Usuarios con 2-5 eventos: {((events_per_user >= 2) & (events_per_user <= 5)).sum()}")
print(f"- Usuarios con 6-10 eventos: {((events_per_user >= 6) & (events_per_user <= 10)).sum()}")
print(f"- Usuarios con más de 10 eventos: {(events_per_user > 10).sum()}")

# Tipos de eventos únicos
print(f"\nTipos de eventos disponibles:")
print(events['event_name'].value_counts())


=== CARACTERIZACIÓN DEL COMPORTAMIENTO DE USUARIOS ===
Distribución de eventos por usuario:
- Usuarios con 1 evento: 759
- Usuarios con 2-5 eventos: 19194
- Usuarios con 6-10 eventos: 28034
- Usuarios con más de 10 eventos: 10716

Tipos de eventos disponibles:
event_name
login           182465
product_page    120862
purchase         60314
product_cart     60120
Name: count, dtype: int64


In [22]:
# Analizar los eventos únicos y su frecuencia
event_counts = events['event_name'].value_counts()
print("Eventos por frecuencia:")
print(event_counts)

Eventos por frecuencia:
event_name
login           182465
product_page    120862
purchase         60314
product_cart     60120
Name: count, dtype: int64


Hay más usuarios en la etapa de purchase, que en la de product_cart, esto es contraintuitivo.

In [26]:
# Count unique users in your dataset
events['user_id'].nunique()

# Count unique events (this should be the event_name column)
events['event_name'].nunique()

# How many unique users made purchases?
purchase_users = events[events['event_name'] == 'purchase']['user_id'].nunique()

# How many unique users added items to cart?
cart_users = events[events['event_name'] == 'product_cart']['user_id'].nunique()

# Compare the numbers
print(f"Users who purchased: {purchase_users}")
print(f"Users who added to cart: {cart_users}")


Users who purchased: 19568
Users who added to cart: 19284


In [27]:
print(events.columns.tolist())
print(events['event_name'].unique())

['user_id', 'event_dt', 'event_name', 'details']
['purchase' 'product_cart' 'product_page' 'login']


In [28]:
# Obtener los conjuntos de usuarios (no solo contar)
purchase_users_set = events[events['event_name'] == 'purchase']['user_id'].unique()
cart_users_set = events[events['event_name'] == 'product_cart']['user_id'].unique()

print(f"Usuarios que compraron: {len(purchase_users_set)}")
print(f"Usuarios que agregaron al carrito: {len(cart_users_set)}")

Usuarios que compraron: 19568
Usuarios que agregaron al carrito: 19284


In [31]:
# Usuarios que compraron pero NO agregaron al carrito
purchase_only = purchase_set - cart_set

# Usuarios que agregaron al carrito pero NO compraron
cart_only = cart_set - purchase_set

print(f"Usuarios que compraron sin agregar al carrito: {len(purchase_only)}")
print(f"Usuarios que agregaron al carrito sin comprar: {len(cart_only)}")

Usuarios que compraron sin agregar al carrito: 13169
Usuarios que agregaron al carrito sin comprar: 12885


13,169 usuarios compraron sin agregar al carrito: Estos usuarios van directamente a la compra, posiblemente porque ya saben exactamente qué quieren o usan funciones como "comprar ahora"

In [32]:
# Ver todos los tipos de eventos únicos en tu dataset
print("Eventos únicos en el dataset:")
print(events['event_name'].unique())
print(f"\nTotal de eventos únicos: {events['event_name'].nunique()}")

Eventos únicos en el dataset:
['purchase' 'product_cart' 'product_page' 'login']

Total de eventos únicos: 4


¿Hay eventos como "buy_now", "direct_purchase", "quick_buy" o algo similar? NO, cómo se explica que hayan mas usuarios en purchase que en add to cart

Analicemos las posibles causas:

Posibles explicaciones:
1. Compras repetidas
- Un usuario puede agregar al carrito una vez, pero hacer múltiples compras
- ¿Has verificado si estás contando usuarios únicos o eventos totales?

2. Diferentes rutas de compra
- Compras desde listas de deseos guardadas
- Compras desde recomendaciones personalizadas
- Compras desde notificaciones push o emails

3. Problemas en el tracking de eventos
- Algunos eventos de "add_to_cart" podrían no estar siendo registrados correctamente
- Diferencias en la implementación del tracking entre páginas

In [38]:
# Confirma que estás contando usuarios únicos
print("Usuarios únicos que compraron:", events[events['event_name'] == 'purchase']['user_id'].nunique())
print("Usuarios únicos que agregaron al carrito:", events[events['event_name'] == 'product_cart']['user_id'].nunique())

Usuarios únicos que compraron: 19568
Usuarios únicos que agregaron al carrito: 19284


In [40]:
# ¿Cuántos eventos de cada tipo hay en total?
print("Total eventos de compra:", len(events[events['event_name'] == 'purchase']))
print("Total eventos de agregar al carrito:", len(events[events['event_name'] == 'product_cart']))

Total eventos de compra: 60314
Total eventos de agregar al carrito: 60120


Fusión de los datasets

In [41]:
# Hacer merge entre events y participants
merged_data = events.merge(participants, on='user_id', how='inner')
print("Forma del DataFrame combinado:", merged_data.shape)
print("\nPrimeras filas:")
print(merged_data.head())
print("\nColumnas disponibles:")
print(merged_data.columns.tolist())

Forma del DataFrame combinado: (102838, 6)

Primeras filas:
            user_id            event_dt event_name  details group  \
0  96F27A054B191457 2020-12-07 04:02:40   purchase     4.99     B   
1  831887FE7F2D6CBA 2020-12-07 06:50:29   purchase     4.99     A   
2  A92195E3CFB83DBD 2020-12-07 00:32:07   purchase     4.99     A   
3  354D653172FF2A2D 2020-12-07 15:45:11   purchase     4.99     A   
4  7FCD34F47C13A9AC 2020-12-07 22:06:13   purchase     9.99     B   

                   ab_test  
0        interface_eu_test  
1  recommender_system_test  
2        interface_eu_test  
3        interface_eu_test  
4        interface_eu_test  

Columnas disponibles:
['user_id', 'event_dt', 'event_name', 'details', 'group', 'ab_test']


In [42]:
# Ver la distribución de grupos por test
print("Distribución de grupos por test:")
print(merged_data.groupby(['ab_test', 'group']).size())

Distribución de grupos por test:
ab_test                  group
interface_eu_test        A        40078
                         B        38851
recommender_system_test  A        18627
                         B         5282
dtype: int64


Interface EU Test:
- Grupo A: 40,078 usuarios
- Grupo B: 38,851 usuarios
- Diferencia: ~1,200 usuarios (3% de diferencia)

Recommender System Test:
- Grupo A: 18,627 usuarios  
- Grupo B: 5,282 usuarios
- Diferencia: ~13,300 usuarios (¡72% de diferencia!)

In [43]:
print("=== ANÁLISIS DE USUARIOS ÚNICOS Y SOLAPAMIENTO PARA 'recommender_system_test' ===")

# Filtrar los datos para el test 'recommender_system_test'
recommender_test_df = merged_data[merged_data['ab_test'] == 'recommender_system_test']

# Obtener usuarios únicos para el Grupo A
users_group_A_recommender = recommender_test_df[recommender_test_df['group'] == 'A']['user_id'].unique()

# Obtener usuarios únicos para el Grupo B
users_group_B_recommender = recommender_test_df[recommender_test_df['group'] == 'B']['user_id'].unique()

print(f"\nUsuarios únicos en Grupo A (recommender_system_test): {len(users_group_A_recommender)}")
print(f"Usuarios únicos en Grupo B (recommender_system_test): {len(users_group_B_recommender)}")

# Verificar solapamiento
overlap_recommender = set(users_group_A_recommender).intersection(set(users_group_B_recommender))

print(f"\nNúmero de usuarios solapados entre Grupo A y Grupo B (recommender_system_test): {len(overlap_recommender)}")

if len(overlap_recommender) > 0:
    print("\n¡Advertencia! Se ha detectado solapamiento de usuarios entre los grupos A y B del test 'recommender_system_test'.")
else:
    print("\nNo se detectó solapamiento de usuarios entre los grupos A y B del test 'recommender_system_test'.")

=== ANÁLISIS DE USUARIOS ÚNICOS Y SOLAPAMIENTO PARA 'recommender_system_test' ===

Usuarios únicos en Grupo A (recommender_system_test): 2747
Usuarios únicos en Grupo B (recommender_system_test): 928

Número de usuarios solapados entre Grupo A y Grupo B (recommender_system_test): 0

No se detectó solapamiento de usuarios entre los grupos A y B del test 'recommender_system_test'.


In [44]:
# 1) Filtrado del test
recommender = merged_data[merged_data['ab_test'] == 'recommender_system_test']

# 2) Número de grupos distintos por usuario
assign_per_user = (recommender
                   .groupby('user_id')['group']
                   .nunique()
                   .reset_index(name='n_groups'))

# 3) Usuarios con más de una asignación de grupo
users_with_multiple_groups = assign_per_user[assign_per_user['n_groups'] > 1]['user_id']

print(f"Usuarios con >1 grupo en este test: {users_with_multiple_groups.nunique()}")

# 4) Inspección rápida de casos problemáticos (si existen)
if not users_with_multiple_groups.empty:
    bad_rows = recommender[recommender['user_id'].isin(users_with_multiple_groups)][['user_id','group']].drop_duplicates()
    print(bad_rows.head(20))


Usuarios con >1 grupo en este test: 0


In [47]:
# Suponiendo que tienes 'enrollment_date' (fecha de inscripción al test)
recommender = merged_data[merged_data['ab_test'] == 'recommender_system_test'].copy()

# Si no tienes 'enrollment_date', usa la fecha mínima de evento por usuario como aproximación
if 'enrollment_date' not in recommender.columns:
    first_event = (recommender.groupby('user_id')['event_dt']
                             .min()
                             .reset_index()
                             .rename(columns={'event_dt':'enrollment_date'}))
    recommender = recommender.merge(first_event, on='user_id', how='left')

# Distribución A/B por día de inscripción
daily_split = (recommender.drop_duplicates(['user_id','group'])  # 1 fila por usuario y grupo
                         .groupby([recommender['enrollment_date'].dt.date, 'group'])
                         .size()
                         .unstack(fill_value=0)
                         .assign(total=lambda df: df.sum(axis=1),
                                 pct_B=lambda df: df.get('B', 0) / df['total'])
              )

print(daily_split)


group              A    B  total     pct_B
enrollment_date                           
2020-12-07       154  173    327  0.529052
2020-12-08       102   47    149  0.315436
2020-12-09        84   88    172  0.511628
2020-12-10        67   33    100  0.330000
2020-12-11        93   21    114  0.184211
2020-12-12        65   53    118  0.449153
2020-12-13        56   17     73  0.232877
2020-12-14       387   72    459  0.156863
2020-12-15       208   33    241  0.136929
2020-12-16       169   99    268  0.369403
2020-12-17       212   39    251  0.155378
2020-12-18       220   50    270  0.185185
2020-12-19       239   47    286  0.164336
2020-12-20       274   63    337  0.186944
2020-12-21       417   93    510  0.182353


In [49]:
# Determinar la fecha de inscripción (primer evento) de cada usuario
user_enrollment = recommender.groupby('user_id')['event_dt'].min().reset_index()
user_enrollment.columns = ['user_id', 'event_dt']

print("Primeros usuarios y sus fechas de inscripción:")
print(user_enrollment.head())

Primeros usuarios y sus fechas de inscripción:
            user_id            event_dt
0  001064FEAAB631A1 2020-12-20 14:43:27
1  0010A1C096941592 2020-12-17 21:07:27
2  00341D8401F0F665 2020-12-21 11:14:50
3  003DF44D7589BBD4 2020-12-17 06:17:49
4  00505E15A9D81546 2020-12-14 17:28:32


In [53]:
# Agregar la fecha de inscripción al dataset principal
df_with_enrollment = recommender.merge(user_enrollment, on='user_id', how='left')

# Calcular días desde la inscripción
df_with_enrollment['days_since_enrollment'] = (
    pd.to_datetime(df_with_enrollment['enrollment_date']) -
    pd.to_datetime(df_with_enrollment['enrollment_date'])
).dt.days

# Filtrar solo los primeros 14 días
df_14_days = df_with_enrollment[df_with_enrollment['days_since_enrollment'] <= 14]

print(f"Datos originales: {len(df)} filas")
print(f"Datos filtrados (14 días): {len(df_14_days)} filas")


Datos originales: 14525 filas
Datos filtrados (14 días): 23909 filas


In [54]:
# Analizar distribución por día de inscripción
daily_distribution = df_14_days.groupby(['enrollment_date', 'group']).size().unstack(fill_value=0)
daily_distribution['total'] = daily_distribution.sum(axis=1)
daily_distribution['pct_B'] = daily_distribution['B'] / daily_distribution['total']

print("Distribución diaria A/B:")
print(daily_distribution)

Distribución diaria A/B:
group                 A   B  total  pct_B
enrollment_date                          
2020-12-07 00:05:57   0   6      6    1.0
2020-12-07 00:07:47   0   8      8    1.0
2020-12-07 00:14:01   4   0      4    0.0
2020-12-07 00:16:00  15   0     15    0.0
2020-12-07 00:25:02   0  24     24    1.0
...                  ..  ..    ...    ...
2020-12-21 23:45:36   3   0      3    0.0
2020-12-21 23:46:42   9   0      9    0.0
2020-12-21 23:53:11   6   0      6    0.0
2020-12-21 23:53:54  10   0     10    0.0
2020-12-21 23:58:57   0   3      3    1.0

[3667 rows x 4 columns]


In [55]:
# Verificar si hay usuarios duplicados
print("Número de usuarios únicos:", df['user_id'].nunique())
print("Número total de filas:", len(df))
print("Promedio de eventos por usuario:", len(df) / df['user_id'].nunique())

# Ver algunos ejemplos de usuarios con múltiples eventos
user_events = df['user_id'].value_counts()
print("\nUsuarios con más eventos:")
print(user_events.head())

Número de usuarios únicos: 13638
Número total de filas: 14525
Promedio de eventos por usuario: 1.0650388620032263

Usuarios con más eventos:
user_id
329E3207708990E1    2
DC4E5D8BD6ABF9BC    2
6B1D0D8C4F8FBBEC    2
DD4731E5B4C9C4EB    2
9918D1C354A1383A    2
Name: count, dtype: int64


Análisis de la estructura de datos:
13,638 usuarios únicos en 14,525 filas
Promedio de 1.065 eventos por usuario (la mayoría tiene 1 evento, algunos tienen 2)
Esto explica por qué el filtrado de 14 días aumentó las filas: cada evento de un usuario se mantiene si está dentro de los 14 días desde su inscripción
El problema real: Asignación temporal de grupos
Lo más preocupante es el patrón que vemos en la distribución A/B. Observa estos momentos específicos:

2020-12-07 00:05:57: 100% grupo B (6 usuarios)
2020-12-07 00:14:01: 100% grupo A (4 usuarios)
2020-12-07 00:25:02: 100% grupo B (24 usuarios)
Esto sugiere que la asignación de grupos no es aleatoria, sino que cambia por bloques de tiempo.

Siguiente paso: Analizar el patrón temporal
Vamos a agrupar por fecha (sin hora) para ver el patrón más claramente:

In [57]:
# Extraer solo la fecha (sin hora) para análisis diario
df_with_enrollment['enrollment_date'] = pd.to_datetime(df_with_enrollment['enrollment_date']).dt.date

# Distribución por día completo
daily_summary = df_14_days.groupby(['enrollment_date', 'group']).size().unstack(fill_value=0)
daily_summary['total'] = daily_summary.sum(axis=1)
daily_summary['pct_B'] = daily_summary['B'] / daily_summary['total']

print("Distribución A/B por día:")
print(daily_summary)

Distribución A/B por día:
group                 A   B  total  pct_B
enrollment_date                          
2020-12-07 00:05:57   0   6      6    1.0
2020-12-07 00:07:47   0   8      8    1.0
2020-12-07 00:14:01   4   0      4    0.0
2020-12-07 00:16:00  15   0     15    0.0
2020-12-07 00:25:02   0  24     24    1.0
...                  ..  ..    ...    ...
2020-12-21 23:45:36   3   0      3    0.0
2020-12-21 23:46:42   9   0      9    0.0
2020-12-21 23:53:11   6   0      6    0.0
2020-12-21 23:53:54  10   0     10    0.0
2020-12-21 23:58:57   0   3      3    1.0

[3667 rows x 4 columns]


In [59]:
# Verificar el formato de la columna de fecha
print("Tipo de dato de enrollment_date:")
print(df_with_enrollment['enrollment_date'].dtype)
print("\nPrimeros valores:")
print(df_with_enrollment['enrollment_date'].head())

# Crear correctamente la fecha sin hora
df_with_enrollment['date_only'] = pd.to_datetime(df_with_enrollment['enrollment_date']).dt.date

print("\nPrimeras fechas sin hora:")
print(df_with_enrollment['date_only'].head())

# Distribución por día completo (corregida)
daily_summary = df_with_enrollment.groupby(['date_only', 'group']).size().unstack(fill_value=0)
daily_summary['total'] = daily_summary.sum(axis=1)
daily_summary['pct_B'] = daily_summary['B'] / daily_summary['total']

print("Distribución A/B por día (corregida):")
print(daily_summary.head(10))  # Solo las primeras 10 filas para ver el patrón


Tipo de dato de enrollment_date:
object

Primeros valores:
0    2020-12-07
1    2020-12-07
2    2020-12-07
3    2020-12-07
4    2020-12-07
Name: enrollment_date, dtype: object

Primeras fechas sin hora:
0    2020-12-07
1    2020-12-07
2    2020-12-07
3    2020-12-07
4    2020-12-07
Name: date_only, dtype: object
Distribución A/B por día (corregida):
group          A     B  total     pct_B
date_only                              
2020-12-07  1038  1327   2365  0.561099
2020-12-08   703   269    972  0.276749
2020-12-09   572   548   1120  0.489286
2020-12-10   354   148    502  0.294821
2020-12-11   566    80    646  0.123839
2020-12-12   318   293    611  0.479542
2020-12-13   269    48    317  0.151420
2020-12-14  3044   358   3402  0.105232
2020-12-15  1572   155   1727  0.089751
2020-12-16  1208   701   1909  0.367208


**Análisis de los resultados**

Mirando tu tabla de distribución diaria, vemos varios patrones preocupantes:


- Días con distribución muy desbalanceada:

2020-12-08: Solo 27.7% en grupo B (debería estar cerca del 50%)
2020-12-11: Solo 12.4% en grupo B
2020-12-13: Solo 15.1% en grupo B
2020-12-14: Solo 10.5% en grupo B
2020-12-15: Solo 9.0% en grupo B
- Días con distribución más equilibrada:

2020-12-07: 56.1% en grupo B
2020-12-09: 48.9% en grupo B
2020-12-12: 48.0% en grupo B

**Principales Problemas Identificados:**
1. Distribución Desbalanceada de Grupos

Encontramos que algunos días tenían solo 10-27% de usuarios en el grupo B, cuando debería ser aproximadamente 50/50
Esta distribución desigual puede sesgar completamente los resultados de la prueba

2. Problemas de Randomización

La asignación aleatoria de usuarios a los grupos no funcionó correctamente
Esto viola uno de los principios fundamentales de las pruebas A/B

3. Variaciones Diarias Extremas

Los patrones de distribución cambiaban drásticamente de un día a otro
Esto sugiere problemas técnicos en el sistema de división del tráfico
¿Por Qué Estos Problemas Son Críticos?
Antes de continuar con el análisis, reflexiona sobre estas preguntas:



In [60]:

# Filtrar eventos de compra
purchase_events = events[events['event_name'] == 'purchase']

# Unir participantes con sus compras
participants_with_purchases = participants.merge(
    purchase_events[['user_id']],
    on='user_id',
    how='left',
    indicator=True
)

# Marcar quién tuvo éxito (hizo al menos una compra)
participants_with_purchases['success'] = (
    participants_with_purchases['_merge'] == 'both'
).astype(int)

In [61]:
# Agrupar por grupo del test
results_by_group = participants_with_purchases.groupby('group').agg({
    'success': ['sum', 'count']
}).round(4)

print("Resultados por grupo:")
print(results_by_group)

Resultados por grupo:
      success       
          sum  count
group               
A        8619  14030
B        6235  10508


In [62]:
# Obtener valores para el z-test
group_stats = participants_with_purchases.groupby('group')['success'].agg(['sum', 'count'])

éxitos_grupo_A = group_stats.loc['A', 'sum']
total_usuarios_A = group_stats.loc['A', 'count']
éxitos_grupo_B = group_stats.loc['B', 'sum']
total_usuarios_B = group_stats.loc['B', 'count']

print(f"Grupo A: {éxitos_grupo_A} éxitos de {total_usuarios_A} usuarios")
print(f"Grupo B: {éxitos_grupo_B} éxitos de {total_usuarios_B} usuarios")

Grupo A: 8619 éxitos de 14030 usuarios
Grupo B: 6235 éxitos de 10508 usuarios


Calculemos las tasas de conversión:

Grupo A: 8,619 ÷ 14,030 = 0.6143 (61.43%)
Grupo B: 6,235 ÷ 10,508 = 0.5935 (59.35%)

Observaciones importantes:

Diferencia en tasas de conversión: El Grupo A tiene una tasa de conversión aproximadamente 2.08 puntos porcentuales más alta que el Grupo B.
Desequilibrio en tamaños de muestra: Como mencionaste anteriormente, hay un problema de randomización. El Grupo A tiene 14,030 usuarios vs 10,508 en el Grupo B. Esto representa una diferencia del ~25%, cuando deberían estar balanceados cerca del 50-50%.

**Análisis estadístico**

Hipótesis nula (H₀): No hay diferencia en las tasas de conversión entre el Grupo A y el Grupo B (π₁ = π₂)

Hipótesis alternativa (H₁): Sí hay diferencia en las tasas de conversión entre los grupos (π₁ ≠ π₂)

Esto es una prueba de dos colas porque estás buscando cualquier diferencia, no específicamente si un grupo es mejor que el otro.

Sobre el desequilibrio de grupos
Tienes razón al preocuparte por esto. El desequilibrio que observaste (14,030 vs 10,508 usuarios) representa aproximadamente una división 57%-43% en lugar del 50%-50% ideal.

Pregunta para reflexionar: ¿Qué factores crees que podrían haber causado este desequilibrio en la asignación de usuarios? ¿Podría esto indicar problemas en el proceso de randomización?

Sobre el nivel de significancia α = 0.05
Perfecto. Esto significa que estás dispuesto a aceptar un 5% de probabilidad de cometer un error Tipo I (rechazar H₀ cuando es verdadera).

In [65]:
import numpy as np
from scipy import stats

# Tus datos específicos
visitors_a = 14030
orders_a = 8619
conversion_rate_a = orders_a / visitors_a

visitors_b = 10508
orders_b = 6235
conversion_rate_b = orders_b / visitors_b

print(f"Tasa de conversión Grupo A: {conversion_rate_a:.4f}")
print(f"Tasa de conversión Grupo B: {conversion_rate_b:.4f}")
# Proporción combinada
p_combined = (orders_a + orders_b) / (visitors_a + visitors_b)

# Error estándar
se = np.sqrt(p_combined * (1 - p_combined) * (1/visitors_a + 1/visitors_b))

# Estadístico z
z_stat = (conversion_rate_a - conversion_rate_b) / se

# Valor p (prueba de dos colas)
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# Nivel de significancia
alpha = 0.05

print("=== RESULTADOS DEL A/B TEST ===")
print(f"Grupo A: {conversion_rate_a:.2%} de conversión")
print(f"Grupo B: {conversion_rate_b:.2%} de conversión")
print(f"Diferencia: {abs(conversion_rate_a - conversion_rate_b):.2%}")
print(f"Estadístico z: {z_stat:.4f}")
print(f"Valor p: {p_value:.4f}")
print(f"Nivel de significancia: {alpha}")
if p_value < alpha:
    print(f"\n✅ RESULTADO: Diferencia estadísticamente significativa")
    print(f"Rechazamos H0. Hay evidencia de diferencia entre los grupos.")
else:
    print(f"\n❌ RESULTADO: No hay diferencia estadísticamente significativa")
    print(f"No rechazamos H0. No hay evidencia suficiente de diferencia.")


Tasa de conversión Grupo A: 0.6143
Tasa de conversión Grupo B: 0.5934
=== RESULTADOS DEL A/B TEST ===
Grupo A: 61.43% de conversión
Grupo B: 59.34% de conversión
Diferencia: 2.10%
Estadístico z: 3.3253
Valor p: 0.0009
Nivel de significancia: 0.05

✅ RESULTADO: Diferencia estadísticamente significativa
Rechazamos H0. Hay evidencia de diferencia entre los grupos.


**CONCLUSIONES DEL ANÁLISIS COMPLETO**
1. Análisis Exploratorio de Datos (EDA)
Principales hallazgos del EDA:

Estructura de datos: 13,638 usuarios únicos con 1.065 eventos promedio por usuario
Desequilibrio en grupos: Distribución desigual entre grupos (14,030 vs 10,508 usuarios)
Patrones temporales: Identificaste asignaciones problemáticas con 100% de usuarios en un solo grupo en ciertos momentos
Calidad de datos: Detectaste la necesidad de agrupar por fecha en lugar de timestamp para análisis temporal correcto

2. Resultados de la Prueba A/B
Métricas principales:
- Grupo A: 61.43% conversión (8,619/14,030)
- Grupo B: 59.34% conversión (6,235/10,508)
- Diferencia: 2.10% a favor del Grupo A

Significancia estadística:
- Z-estadístico: 3.33
- Valor p: 0.0009
- Conclusión: Diferencia estadísticamente significativa (p < 0.05)

3. Interpretación Integral
¿Qué te dicen estos resultados juntos?

El EDA reveló problemas de implementación que podrían afectar la validez del experimento
A pesar de los problemas, la diferencia es robusta (z = 3.33 es bastante fuerte)
El desequilibrio en tamaños de muestra no impidió detectar la diferencia

**Recomendación Principal:**
"No se recomienda implementar cambios basados en los resultados de esta prueba A/B debido a múltiples problemas metodológicos que comprometen la validez y confiabilidad de los hallazgos."

Justificación Detallada
1. Problemas de Calidad de Datos:

Desbalance significativo en los tamaños de grupo (diferencia del 4% aproximadamente).
Presencia de anomalías que distorsionan las métricas clave.
Inconsistencias en los datos que sugieren problemas en la implementación.

2. Limitaciones Metodológicas:

La significancia estadística no garantiza relevancia práctica para el negocio.
Los resultados muestran alta variabilidad y falta de estabilización.
El análisis revela patrones inconsistentes entre métricas

3. Riesgos para el Negocio:

Implementar cambios basados en datos poco confiables podría resultar en pérdidas económicas.
La incertidumbre en los resultados hace imposible predecir el impacto real.
Los costos de implementación no justifican el nivel de confianza actual.